# Phase 4: Modeling

This phase covers model training, hyperparameter tuning, and model evaluation using scikit-learn and XGBoost.


In [1]:
# Re-import dependencies for this phase
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

from src.processing import load_data, preprocess_data, engineer_features, prepare_modeling_data
from src.modeling import train_models, evaluate_all_models, metrics_summary_df, get_best_model, tune_hyperparameters, evaluate_model

# Reload data through the full pipeline
df = load_data()
df = preprocess_data(df)
df_feat = engineer_features(df)
print(f'Engineered dataset: {df_feat.shape[0]:,} rows x {df_feat.shape[1]} columns')


Dataset loaded: 90,189 rows × 5 columns
Capped 898 extreme game-round values at 99% percentile (493 rounds).
After cleaning: 90,189 rows
Engineered features added. New shape: (90189, 9)
Train: 72,151  |  Test: 18,038
Positive-class rate (train): 0.1861
Features (6): ['sum_gamerounds', 'version', 'retention_1', 'high_engagement', 'retention_1_x_rounds', 'rounds_per_day_proxy']
Training set: (72151, 6), Test set: (18038, 6)


---
## Step 4 - Train / Test Split

In [5]:
from src.processing import prepare_modeling_data

X_train, X_test, y_train, y_test, feature_names = prepare_modeling_data(df_feat)
print(f"Feature names: {feature_names}")


NameError: name 'df_feat' is not defined

---
## Step 5 - Model Training (sklearn Pipeline)

Three models via `imblearn.Pipeline` with SMOTE:
- **Logistic Regression** - baseline
- **Random Forest** - ensemble
- **XGBoost** - gradient boosting

In [ ]:
from src.modeling import train_models

trained_models = train_models(X_train, y_train)
print(f"\n✓ {len(trained_models)} models trained")

  Training Logistic Regression …
  Training Random Forest …
  Training XGBoost …
  Training Gradient Boosting …

✓ 4 models trained


---
## Step 6 - Hyperparameter Tuning

GridSearchCV with 5-fold stratified CV.

In [ ]:
from src.modeling import tune_hyperparameters

tuned_xgb = tune_hyperparameters(X_train, y_train, model_name='XGBoost')
tuned_rf = tune_hyperparameters(X_train, y_train, model_name='Random Forest')

Tuning XGBoost (5-fold CV, scoring=roc_auc) …
  Best params : {'classifier__learning_rate': 0.1, 'classifier__max_depth': 3, 'classifier__n_estimators': 50}
  Best CV roc_auc: 0.8892
Tuning Random Forest (5-fold CV, scoring=roc_auc) …
  Best params : {'classifier__max_depth': 5, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 200}
  Best CV roc_auc: 0.8889


---
## Summary

| Step | Action | Output |
|------|--------|--------|
| Features | `engineer_features()` | Interaction terms, bins |
| Split | `prepare_modeling_data()` | 80/20 stratified |
| Train | `train_models()` | LR, RF, XGBoost |
| Tune | `tune_hyperparameters()` | Best params via GridSearchCV |

-> Next: `06_evaluation_and_ab_testing.ipynb`